# **Generating adversarial noise examples using FGSM**
By Charlot Eberlein, Loughborough University • 11.01.26 • 
[LinkedIn](https://www.linkedin.com/in/charloteberlein/) • 
[Portfolio](https://charloteberlein.github.io) • 
[GitHub](https://github.com/charloteberlein)

In [1]:
%%capture
import keras as k
import matplotlib.pyplot as plt
import numpy as np

### **Step 1:** Load the MNIST dataset

In [7]:
(train_im, train_l), (test_im, test_l) = k.datasets.mnist.load_data()

train_im = train_im.reshape((60000,28,28,1)).astype("float32")/255
test_im = test_im.reshape((10000,28,28,1)).astype("float32")/255

Let's quickly make sure it works using matplotlib:

In [ ]:
SIZE = 6; assert SIZE > 0
fig, ax = plt.subplots(SIZE, SIZE)
fig.suptitle(f"First {SIZE**2} MNIST digits")
for i in range(SIZE):
    for j in range(SIZE):
        p = SIZE*i + j
        ax[i][j].imshow(np.reshape(train_im[p], (28,28)), cmap="gray")
        ax[i][j].set_title(train_l[p], y=-0.1, x=0.9, color="white")
        ax[i][j].set_axis_off()
plt.tight_layout()
plt.show()

If the dataset has been loaded correctly, the result should look something like this:

![Grid of digits](resources/first_36_mnist.png)

Great. Now, let's move on to the next step.

### **Step 2:** Build a Convolutional Neural Network (CNN) with TensorFlow
The architecture we're going to use is as follows:
- Input layer
- Data augmentation layer (random elastic transform)
- Convolutional layer (ReLU activation)
- Batch normalisation
- Max-pooling layer
- Convolutional layer (ReLU activation)
- Batch normalisation
- Max-pooling layer
- Fully-connected layer
- Output layer (Softmax activation)

**Random elastic transform** is a data augmentation method I think is best described like the "liquify" tool in photoshop. Here, we have an intensity of 0.1 which is randomly applied in 5% of the training data.

**Batch normalisation** normalises layer inputs to a mean of 0 and standard deviation of 1, which improves training speed and stability.

In [4]:
model = k.Sequential()
model.add(k.Input(shape=(28,28,1)))
model.add(k.layers.RandomElasticTransform(factor=0.05, scale=0.1))
model.add(k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.BatchNormalization())
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=4))
model.add(k.layers.Conv2D(64, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.BatchNormalization())
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=4))
model.add(k.layers.Dense(128, activation="relu"))
model.add(k.layers.Flatten())
model.add(k.layers.Dense(10, activation="softmax"))
# compile the finished model
model.compile(optimizer="sgd", loss="categorical_crossentropy",
              metrics=["accuracy"])

#### **Optional:** Train the model
Because this is a really simple problem, training should only take a minute or two. However, I kept the weights from training it earlier, so you can skip this step.

**Training accuracy:** 97%\
**Validation accuracy:** 99%

In [ ]:
# re-format labels
train_l = k.utils.to_categorical(train_l)
test_l = k.utils.to_categorical(test_l)

result = model.fit(train_im, train_l, batch_size=64, epochs=16, verbose=1,
                   validation_data=(test_im,test_l))
model.save_weights("resources/mnist-nn.weights.h5")

fig, ax = plt.subplots()
ax.plot(result.history['loss'], color="b")
ax.plot(result.history['val_loss'], color="r")
ax.set_title("Training and validation loss")
ax.set_xlabel("Epochs")
ax.set_ylabel("Loss")
ax.set_xticks(np.arange(0,16))
ax.legend(("Training loss", "Validation loss"))
plt.show()

If you chose to train it, you'd get something like the graph below, although neural network training can be quite variable.

We use loss curves to help us identify after how many epochs we should stop training. Ideally, we want a nice smooth plateau; if the curves haven't converged yet, that means the model is underfitted, and if the loss is starting to rise again or is oscillating a lot, then that means the model is overfitted.

The distance between the curves on this graph isn't due to underfitting, though, but rather the augmentation that we added to the training data.

![Loss-Epochs curves, showing two lines in exponential decay](resources/loss_epochs_data_augmentation.png)

An ideal loss-epoch curve looks a bit more like this. The only difference between this model and the model we're using is that this one didn't have any kind of data augmentation. So, it has a nicer curve, but the actual model accuracy is lower.

![Loss-Epochs curves, showing two lines very close in exponential decay](resources/loss_epochs_curve.png)

### **Step 3:** Use FGSM to generate Adversarial Noise Examples